# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is an object, access fields as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate all available record sets in the dataset schema, showing their `@id`, and the `@id`s of their fields (columns).

In [ ]:
# Find all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    # Each record set has fields: list of Field objects
    fields = rs.get('fields', [])
    print(f"  Fields in record set:")
    for f in fields:
        print(f"    {f['@id']} - {f.get('name', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
# First, construct the list of @id for available record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Now display columns for one record set (use the first as example)
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"\nColumns in record set {preview_id}:\n{dataframes[preview_id].columns.tolist()}")
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick the main record set to analyze (using the first one here)
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id]

# List numeric fields
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields available: {numeric_fields}")

# Choose the first numeric field (adapt as needed)
if numeric_fields:
    numeric_field = numeric_fields[0]  # for illustration
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as a threshold example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical/text field
    group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = None
    for field in group_fields:
        if field != numeric_field:  # avoid grouping by numeric field
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a second numeric field exists, plot a scatter
    if len(numeric_fields) > 1:
        other_numeric = numeric_fields[1]
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=numeric_field, y=other_numeric)
        plt.title(f"{numeric_field} vs {other_numeric}")
        plt.xlabel(numeric_field)
        plt.ylabel(other_numeric)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrated loading a Croissant-based dataset (`mlcroissant`), exploring its record sets and fields using `@id` references, extracting records, and conducting exploratory data analysis and visualization.
* The schema is rich and may include multiple types of record sets, fields describing protein measurements, and useful metadata tying the records to their biological meaning.
* You should adapt the notebook to analyze particular features of interest and to perform more advanced statistics as fits your research questions.